In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict
import ast

import numpy as np
import pandas as pd
from pymongo import MongoClient
from pydantic import BaseModel
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

LONG_THRESHOLD = 30.0


In [5]:
class EventsConfig(BaseModel):
    db: str
    url: str
    collection: Dict[str, str]
    year: str


def load_config(path: str | Path = "config/config.yaml") -> EventsConfig:
    path = Path(path)
    if not path.exists():
        path = Path("../config/config.yaml")

    with path.open("r", encoding="utf-8") as f:
        raw = yaml.safe_load(f)

    return EventsConfig(
        db=raw["mongo"]["db"],
        url=raw["mongo"]["url"],
        collection=raw["mongo"]["collection"],
        year=raw["season"]["year"],
    )


events_config = load_config()
events_config


EventsConfig(db='WhoScored', url='mongodb://localhost:27017/', collection={'collection_teams': 'available_teams', 'collection_schedule': 'game_schedule', 'collection_logs': 'error_logs', 'collection_raw_events': 'game_raw_events', 'collection_processed_events': 'game_processed_events', 'collection_team_game_stats': 'game_team_stats', 'collection_player_game_stats': 'game_player_stats'}, year='2009-2010')

In [ ]:
# client = MongoClient(events_config.url)
# db = client[events_config.db]
# db[events_config.collection.get("collection_player_game_stats")].delete_many({})

DeleteResult({'n': 8384, 'ok': 1.0}, acknowledged=True)

## Qualifier Helpers


In [2]:
def _coerce_qualifiers(qs: Any) -> list[Any]:
    if qs is None:
        return []
    if isinstance(qs, list):
        return qs
    if isinstance(qs, tuple):
        return list(qs)
    if isinstance(qs, dict):
        return [qs]
    if isinstance(qs, str):
        text = qs.strip()
        if not text or text.lower() in {"nan", "none", "null"}:
            return []
        try:
            parsed = ast.literal_eval(text)
        except (SyntaxError, ValueError):
            return [text]
        return _coerce_qualifiers(parsed)
    try:
        if pd.isna(qs):
            return []
    except (TypeError, ValueError):
        pass
    return []


def build_qset(qs: Any) -> set[str]:
    out = set()
    for q in _coerce_qualifiers(qs):
        if isinstance(q, str):
            if q:
                out.add(q)
        elif isinstance(q, dict):
            qtype = q.get("type")
            if isinstance(qtype, dict):
                name = qtype.get("displayName") or qtype.get("value")
            else:
                name = q.get("displayName") or q.get("name") or q.get("type")
            if name:
                out.add(str(name))
    return out


def ensure_qual_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "qualifiers" not in df.columns:
        df["qualifiers"] = [[] for _ in range(len(df))]
    df["qset"] = df["qualifiers"].apply(build_qset)
    return df


def has(df: pd.DataFrame, qual: str) -> pd.Series:
    return df["qset"].map(lambda s: qual in s)


def has_any(df: pd.DataFrame, quals) -> pd.Series:
    q = set(quals)
    return df["qset"].map(lambda s: bool(s & q))


In [3]:
def build_qmap(qs: Any) -> dict[str, Any]:
    out = {}
    for q in _coerce_qualifiers(qs):
        if isinstance(q, str):
            if q:
                out[q] = None
        elif isinstance(q, dict):
            qtype = q.get("type")
            if isinstance(qtype, dict):
                name = qtype.get("displayName") or qtype.get("value")
            else:
                name = q.get("displayName") or q.get("name") or q.get("type")
            if name:
                out[str(name)] = q.get("value")
    return out


def _qvalue(df: pd.DataFrame, qual: str) -> pd.Series:
    return df["qmap"].map(lambda m: m.get(qual) if isinstance(m, dict) else None)


def _num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def build_processed_game_events(raw_events: pd.DataFrame) -> pd.DataFrame:
    events = ensure_qual_cols(raw_events).copy()
    events["qmap"] = events["qualifiers"].apply(build_qmap)
    events["qualifier_names"] = events["qset"].map(lambda s: sorted(s))
    events["type"] = events["type"].fillna("")
    events["outcome_type"] = events["outcome_type"].fillna("")

    t = events["type"]
    outcome = events["outcome_type"]
    pass_mask = t.eq("Pass")
    shot_mask = t.isin({"Goal", "MissedShots", "SavedShot", "ShotOnPost", "OwnGoal"})

    # Shot categories.
    shot_block_quals = {"Blocked", "OutfielderBlock", "ShotBlocked", "AttemptBlocked", "SixYardBlock"}
    small_box_quals = {"SmallBoxLeft", "SmallBoxCentre", "SmallBoxRight"}
    box_quals = {"BoxLeft", "BoxCentre", "BoxRight", "DeepBoxLeft", "DeepBoxCentre", "DeepBoxRight"}
    out_box_quals = {
        "OutOfBoxLeft", "OutOfBoxCentre", "OutOfBoxRight",
        "OutOfBoxDeepLeft", "OutOfBoxDeepRight",
        "ThirtyFivePlusLeft", "ThirtyFivePlusCentre", "ThirtyFivePlusRight",
    }
    set_piece_quals = {
        "SetPiece", "FromCorner", "CornerTaken", "ThrowinSetPiece", "ThrowIn", "ThrowInSetPiece",
        "DirectFreeKick", "DirectFreekick", "IndirectFreeKickTaken", "IndirectFreekickTaken",
        "FreekickTaken", "FreeKickTaken",
    }

    own_goal = t.eq("OwnGoal") | has(events, "OwnGoal")
    shot_blocked = shot_mask & has_any(events, shot_block_quals) & ~own_goal
    shot_woodwork = t.eq("ShotOnPost") & ~own_goal
    shot_goal = t.eq("Goal") & ~own_goal
    shot_on_target = t.isin({"Goal", "SavedShot"}) & ~shot_blocked & ~own_goal
    shot_off_target = (t.eq("MissedShots") | shot_woodwork) & ~shot_blocked & ~own_goal

    events["shot_result"] = np.select(
        [own_goal, shot_goal, shot_on_target, shot_off_target, shot_blocked, shot_woodwork],
        ["Own Goal", "Goal", "On Target", "Off Target", "Blocked", "Woodwork"],
        default=None,
    )
    events["shot_zone"] = np.select(
        [has_any(events, small_box_quals), has_any(events, box_quals), has_any(events, out_box_quals)],
        ["6-yard box", "Penalty Area", "Outside of box"],
        default="Unknown",
    )
    events.loc[~shot_mask, "shot_zone"] = None
    shot_penalty = shot_mask & has(events, "Penalty") & ~own_goal
    shot_fastbreak = shot_mask & has(events, "FastBreak") & ~own_goal
    shot_set_piece = shot_mask & has_any(events, set_piece_quals) & ~shot_penalty & ~own_goal
    shot_open_play = shot_mask & has(events, "RegularPlay") & ~shot_penalty & ~shot_fastbreak & ~shot_set_piece & ~own_goal
    events["shot_situation"] = np.select(
        [own_goal, shot_penalty, shot_fastbreak, shot_set_piece, shot_open_play],
        ["Own Goal", "Penalty", "Fastbreak", "Set Pieces", "Open Play"],
        default="Unknown",
    )
    events.loc[~shot_mask, "shot_situation"] = None
    events["shot_body_part"] = np.select(
        [has(events, "RightFoot"), has(events, "LeftFoot"), has(events, "Head")],
        ["Right foot", "Left foot", "Head"],
        default="Other body parts",
    )
    events.loc[~shot_mask, "shot_body_part"] = None
    events["shot_is_goal"] = shot_goal.astype(int)
    events["shot_on_target"] = shot_on_target.astype(int)
    events["shot_off_target"] = shot_off_target.astype(int)
    events["shot_blocked"] = shot_blocked.astype(int)
    events["shot_woodwork"] = shot_woodwork.astype(int)
    events["shot_own_goal"] = own_goal.astype(int)

    # Pass categories. Most categories are pass attempts. Key passes follow WhoScored's shot-assisted display count.
    px = _num(events["x"])
    py = _num(events["y"])
    pex = _num(events["end_x"])
    pey = _num(events["end_y"])
    pdx = pex - px
    pdy = pey - py
    target_zone_1 = 100 / 3
    target_zone_2 = 200 / 3
    freekick_quals = {"FreekickTaken", "FreeKickTaken", "IndirectFreekickTaken", "IndirectFreeKickTaken"}
    long_excluded_type = has_any(events, {"Cross", "KeyPass", "Throughball"})
    pass_long = pass_mask & has(events, "Longball") & ~long_excluded_type

    events["pass_successful"] = (pass_mask & outcome.eq("Successful")).astype(int)
    events["pass_length"] = np.where(pass_long, "Long", "Short")
    events.loc[~pass_mask, "pass_length"] = None
    events["pass_height"] = np.where(has(events, "Chipped"), "Chipped", "Ground")
    events.loc[~pass_mask, "pass_height"] = None
    events["pass_body_part"] = np.where(has(events, "HeadPass"), "Head", "Feet")
    events.loc[~pass_mask, "pass_body_part"] = None
    events["pass_target_zone"] = np.select(
        [pex < target_zone_1, (pex >= target_zone_1) & (pex < target_zone_2), pex >= target_zone_2],
        ["Defensive Third", "Mid Third", "Final Third"],
        default=None,
    )
    events.loc[~pass_mask, "pass_target_zone"] = None
    events["pass_forward"] = (pass_mask & pdx.gt(0)).astype(int)
    events["pass_backward"] = (pass_mask & pdx.lt(0)).astype(int)
    events["pass_left"] = (pass_mask & pdy.gt(0)).astype(int)
    events["pass_right"] = (pass_mask & pdy.lt(0)).astype(int)
    events["pass_cross"] = (pass_mask & has(events, "Cross")).astype(int)
    events["pass_freekick"] = (pass_mask & has_any(events, freekick_quals) & ~has(events, "Cross")).astype(int)
    events["pass_corner"] = (pass_mask & has(events, "CornerTaken")).astype(int)
    events["pass_through_ball"] = (pass_mask & has(events, "Throughball")).astype(int)
    events["pass_throw_in"] = (pass_mask & has(events, "ThrowIn")).astype(int)
    events["pass_key_pass_qualifier"] = (pass_mask & has(events, "KeyPass")).astype(int)
    events["shot_assisted_key_pass"] = (shot_mask & has(events, "Assisted")).astype(int)

    # Defensive and other stat categories.
    outfielder_block = t.eq("Save") & has(events, "OutfielderBlock")
    blocked_cross = t.eq("BlockedPass") | (t.eq("Clearance") & has(events, "BlockedCross"))
    blocked_shot = t.eq("Block") | outfielder_block
    block_mask = blocked_cross | blocked_shot
    tackle_mask = t.eq("Tackle") | (t.eq("Challenge") & outcome.eq("Unsuccessful"))
    clearance_mask = t.eq("Clearance") & ~has(events, "BlockedCross")
    foul_mask = t.eq("Foul")
    aerial_mask = t.eq("Aerial")
    loss_possession_mask = t.eq("Dispossessed") | (t.eq("BallTouch") & outcome.eq("Unsuccessful"))
    error_mask = has_any(events, {"LeadingToAttempt", "LeadingToGoal"})
    card_mask = t.eq("Card")
    gk_mask = (t.eq("Save") & ~has(events, "OutfielderBlock")) | t.isin({"Claim", "Punch", "KeeperPickup", "KeeperSweeper"})
    offside_mask = t.isin({"OffsideGiven", "OffsidePass", "OffsideProvoked"})

    events["tackle_result"] = np.select(
        [t.eq("Tackle") & outcome.eq("Successful"), t.eq("Tackle") & outcome.eq("Unsuccessful"), t.eq("Challenge") & outcome.eq("Unsuccessful")],
        ["Gained Possession", "Did Not Get Possession", "Was Dribbled"],
        default=None,
    )
    events["clearance_body_part"] = np.where(clearance_mask & has(events, "Head"), "Head", "Feet")
    events.loc[~clearance_mask, "clearance_body_part"] = None
    events["block_type"] = np.select([blocked_shot, blocked_cross], ["Blocked Shot", "Blocked Cross"], default=None)
    events["offside_type"] = np.select(
        [t.eq("OffsideGiven"), t.eq("OffsidePass"), t.eq("OffsideProvoked")],
        ["Caught Offside", "Offside Pass", "Offside Provoked"],
        default=None,
    )
    events["foul_type"] = np.select(
        [foul_mask & outcome.eq("Unsuccessful"), foul_mask & outcome.eq("Successful")],
        ["Foul Committed", "Foul Suffered"],
        default=None,
    )
    events["aerial_result"] = np.select(
        [aerial_mask & outcome.eq("Successful"), aerial_mask & outcome.eq("Unsuccessful")],
        ["Won", "Lost"],
        default=None,
    )
    events["loss_possession_type"] = np.select(
        [t.eq("Dispossessed"), t.eq("BallTouch") & outcome.eq("Unsuccessful")],
        ["Dispossessed", "Turnover"],
        default=None,
    )
    events["error_type"] = np.select(
        [has(events, "LeadingToGoal"), has(events, "LeadingToAttempt")],
        ["Error Leading to Goal", "Error Leading to Shot"],
        default=None,
    )
    events["gk_type"] = np.select(
        [t.eq("Save") & ~has(events, "OutfielderBlock"), t.eq("Claim"), t.eq("Punch"), t.eq("KeeperPickup"), t.eq("KeeperSweeper")],
        ["Save", "Claim", "Punch", "Keeper Pickup", "Keeper Sweeper"],
        default=None,
    )
    events["card_type"] = np.select(
        [has(events, "SecondYellow"), has(events, "Yellow"), has(events, "Red")],
        ["Second Yellow", "Yellow", "Red"],
        default=None,
    )
    events["substitution_type"] = np.select([t.eq("SubstitutionOff"), t.eq("SubstitutionOn")], ["Off", "On"], default=None)


    # Row-level stat flags. This is the main Mongo-ready processed game-events shape.
    events["is_shot"] = shot_mask.astype(int)
    events["is_shot_goal"] = events["shot_is_goal"]
    events["is_shot_on_target"] = events["shot_on_target"]
    events["is_shot_off_target"] = events["shot_off_target"]
    events["is_shot_blocked"] = events["shot_blocked"]
    events["is_shot_woodwork"] = events["shot_woodwork"]
    events["is_shot_own_goal"] = events["shot_own_goal"]

    events["is_pass"] = pass_mask.astype(int)
    events["is_pass_attempt"] = pass_mask.astype(int)
    events["is_pass_completed"] = events["pass_successful"]
    events["is_pass_incomplete"] = (pass_mask & ~outcome.eq("Successful")).astype(int)
    events["is_pass_long"] = pass_long.astype(int)
    events["is_pass_short"] = (pass_mask & ~pass_long).astype(int)
    events["is_pass_chipped"] = (pass_mask & has(events, "Chipped")).astype(int)
    events["is_pass_ground"] = (pass_mask & ~has(events, "Chipped")).astype(int)
    events["is_pass_head"] = (pass_mask & has(events, "HeadPass")).astype(int)
    events["is_pass_feet"] = (pass_mask & ~has(events, "HeadPass")).astype(int)
    events["is_pass_forward"] = events["pass_forward"]
    events["is_pass_backward"] = events["pass_backward"]
    events["is_pass_left"] = events["pass_left"]
    events["is_pass_right"] = events["pass_right"]
    events["is_pass_defensive_third"] = (pass_mask & events["pass_target_zone"].eq("Defensive Third")).astype(int)
    events["is_pass_mid_third"] = (pass_mask & events["pass_target_zone"].eq("Mid Third")).astype(int)
    events["is_pass_final_third"] = (pass_mask & events["pass_target_zone"].eq("Final Third")).astype(int)
    events["is_pass_cross"] = events["pass_cross"]
    events["is_pass_freekick"] = events["pass_freekick"]
    events["is_pass_corner"] = events["pass_corner"]
    events["is_pass_through_ball"] = events["pass_through_ball"]
    events["is_pass_throw_in"] = events["pass_throw_in"]
    events["is_pass_key_pass_qualifier"] = events["pass_key_pass_qualifier"]
    events["is_key_pass"] = events["shot_assisted_key_pass"]

    events["is_dribble"] = t.eq("TakeOn").astype(int)
    events["is_dribble_successful"] = (t.eq("TakeOn") & outcome.eq("Successful")).astype(int)
    events["is_dribble_unsuccessful"] = (t.eq("TakeOn") & outcome.eq("Unsuccessful")).astype(int)
    events["is_tackle"] = tackle_mask.astype(int)
    events["is_tackle_gained_possession"] = events["tackle_result"].eq("Gained Possession").astype(int)
    events["is_tackle_did_not_get_possession"] = events["tackle_result"].eq("Did Not Get Possession").astype(int)
    events["is_tackle_was_dribbled"] = events["tackle_result"].eq("Was Dribbled").astype(int)
    events["is_interception"] = t.eq("Interception").astype(int)
    events["is_ball_recovery"] = t.eq("BallRecovery").astype(int)
    events["is_clearance"] = clearance_mask.astype(int)
    events["is_clearance_head"] = (clearance_mask & events["clearance_body_part"].eq("Head")).astype(int)
    events["is_clearance_feet"] = (clearance_mask & events["clearance_body_part"].eq("Feet")).astype(int)
    events["is_block"] = block_mask.astype(int)
    events["is_blocked_shot"] = events["block_type"].eq("Blocked Shot").astype(int)
    events["is_blocked_cross"] = events["block_type"].eq("Blocked Cross").astype(int)
    events["is_offside"] = offside_mask.astype(int)
    events["is_caught_offside"] = events["offside_type"].eq("Caught Offside").astype(int)
    events["is_offside_pass"] = events["offside_type"].eq("Offside Pass").astype(int)
    events["is_offside_provoked"] = events["offside_type"].eq("Offside Provoked").astype(int)
    events["is_foul"] = foul_mask.astype(int)
    events["is_foul_committed"] = events["foul_type"].eq("Foul Committed").astype(int)
    events["is_foul_suffered"] = events["foul_type"].eq("Foul Suffered").astype(int)
    events["is_aerial_duel"] = aerial_mask.astype(int)
    events["is_aerial_duel_won"] = events["aerial_result"].eq("Won").astype(int)
    events["is_aerial_duel_lost"] = events["aerial_result"].eq("Lost").astype(int)
    events["is_loss_possession"] = loss_possession_mask.astype(int)
    events["is_dispossessed"] = events["loss_possession_type"].eq("Dispossessed").astype(int)
    events["is_turnover"] = events["loss_possession_type"].eq("Turnover").astype(int)
    events["is_error"] = error_mask.astype(int)
    events["is_error_leading_to_shot"] = events["error_type"].eq("Error Leading to Shot").astype(int)
    events["is_error_leading_to_goal"] = events["error_type"].eq("Error Leading to Goal").astype(int)
    events["is_goalkeeper"] = gk_mask.astype(int)
    events["is_gk_save"] = events["gk_type"].eq("Save").astype(int)
    events["is_gk_claim"] = events["gk_type"].eq("Claim").astype(int)
    events["is_gk_punch"] = events["gk_type"].eq("Punch").astype(int)
    events["is_gk_keeper_pickup"] = events["gk_type"].eq("Keeper Pickup").astype(int)
    events["is_gk_keeper_sweeper"] = events["gk_type"].eq("Keeper Sweeper").astype(int)

    events = events.copy()
    events["is_card"] = card_mask.astype(int)
    events["is_yellow_card"] = events["card_type"].eq("Yellow").astype(int)
    events["is_second_yellow_card"] = events["card_type"].eq("Second Yellow").astype(int)
    events["is_red_card"] = events["card_type"].eq("Red").astype(int)
    events["is_substitution"] = t.isin({"SubstitutionOff", "SubstitutionOn"}).astype(int)
    events["is_substitution_on"] = events["substitution_type"].eq("On").astype(int)
    events["is_substitution_off"] = events["substitution_type"].eq("Off").astype(int)

    events = events.copy()
    events["stat_event_type"] = None
    stat_type_order = [
        (events["is_shot"].eq(1), "shot"),
        (events["is_pass"].eq(1), "pass"),
        (events["is_dribble"].eq(1), "dribble"),
        (events["is_tackle"].eq(1), "tackle"),
        (events["is_interception"].eq(1), "interception"),
        (events["is_ball_recovery"].eq(1), "ball_recovery"),
        (events["is_clearance"].eq(1), "clearance"),
        (events["is_block"].eq(1), "block"),
        (events["is_offside"].eq(1), "offside"),
        (events["is_foul"].eq(1), "foul"),
        (events["is_aerial_duel"].eq(1), "aerial_duel"),
        (events["is_loss_possession"].eq(1), "loss_possession"),
        (events["is_error"].eq(1), "error"),
        (events["is_goalkeeper"].eq(1), "goalkeeper"),
        (events["is_card"].eq(1), "card"),
        (events["is_substitution"].eq(1), "substitution"),
    ]
    for mask, label in stat_type_order:
        events.loc[mask, "stat_event_type"] = label

    relevant_mask = events["stat_event_type"].notna() | events["is_touch"].eq(1) | events["is_key_pass"].eq(1)
    processed_events = events.loc[relevant_mask].reset_index(drop=True).copy()

    # These raw qualifier helper columns are only needed during processing.
    drop_cols = ["qualifiers", "qset", "qmap", "qualifier_names"]
    processed_events = processed_events.drop(columns=[c for c in drop_cols if c in processed_events.columns])

    def as_bool(source_col: str) -> pd.Series:
        if source_col not in processed_events.columns:
            return pd.Series(False, index=processed_events.index)
        return pd.to_numeric(processed_events[source_col], errors="coerce").fillna(0).astype(int).astype(bool)

    # Final Mongo-ready flags: keep categories as labels, but expose every stat/substat as booleans.
    final_boolean_sources = {
        # WhoScored tile order
        "shot_event": "is_shot",
        "pass_event": "is_pass",
        "pass_attempt": "is_pass_attempt",
        "pass_completed": "pass_successful",
        "pass_incomplete": "is_pass_incomplete",
        "dribble_event": "is_dribble",
        "tackle_attempted_event": "is_tackle",
        "interception_event": "is_interception",
        "clearance_event": "is_clearance",
        "block_event": "is_block",
        "offside_event": "is_offside",
        "foul_event": "is_foul",
        "aerial_duel_event": "is_aerial_duel",
        "touch_event": "is_touch",
        "loss_possession_event": "is_loss_possession",
        "error_event": "is_error",
        "goalkeeper_event": "is_goalkeeper",
        "save_event": "is_gk_save",
        "claim_event": "is_gk_claim",
        "punch_event": "is_gk_punch",
        "ball_recovery_event": "is_ball_recovery",
        "card_event": "is_card",
        "substitution_event": "is_substitution",
        # Shot detail
        "shot_goal": "shot_is_goal",
        "shot_on_target": "shot_on_target",
        "shot_off_target": "shot_off_target",
        "shot_woodwork": "shot_woodwork",
        "shot_blocked": "shot_blocked",
        "shot_own_goal": "shot_own_goal",
        # Pass detail
        "pass_long": "is_pass_long",
        "pass_short": "is_pass_short",
        "pass_chipped": "is_pass_chipped",
        "pass_ground": "is_pass_ground",
        "pass_head": "is_pass_head",
        "pass_feet": "is_pass_feet",
        "pass_forward": "pass_forward",
        "pass_backward": "pass_backward",
        "pass_left": "pass_left",
        "pass_right": "pass_right",
        "pass_defensive_third": "is_pass_defensive_third",
        "pass_mid_third": "is_pass_mid_third",
        "pass_final_third": "is_pass_final_third",
        "pass_cross": "pass_cross",
        "pass_freekick": "pass_freekick",
        "pass_corner": "pass_corner",
        "pass_through_ball": "pass_through_ball",
        "pass_throw_in": "pass_throw_in",
        "pass_key_pass_qualifier": "pass_key_pass_qualifier",
        "pass_key_pass": "is_key_pass",
        # Other detail
        "dribble_successful": "is_dribble_successful",
        "dribble_unsuccessful": "is_dribble_unsuccessful",
        "dispossessed": "is_dispossessed",
        "turnover": "is_turnover",
        "tackle_gained_possession": "is_tackle_gained_possession",
        "tackle_did_not_get_possession": "is_tackle_did_not_get_possession",
        "tackle_was_dribbled": "is_tackle_was_dribbled",
        "clearance_head": "is_clearance_head",
        "clearance_feet": "is_clearance_feet",
        "blocked_shot": "is_blocked_shot",
        "blocked_cross": "is_blocked_cross",
        "caught_offside": "is_caught_offside",
        "offside_pass": "is_offside_pass",
        "offside_provoked": "is_offside_provoked",
        "foul_committed": "is_foul_committed",
        "foul_suffered": "is_foul_suffered",
        "aerial_duel_won": "is_aerial_duel_won",
        "aerial_duel_lost": "is_aerial_duel_lost",
        "error_leading_to_shot": "is_error_leading_to_shot",
        "error_leading_to_goal": "is_error_leading_to_goal",
        "keeper_pickup": "is_gk_keeper_pickup",
        "keeper_sweeper": "is_gk_keeper_sweeper",
        "yellow_card": "is_yellow_card",
        "second_yellow_card": "is_second_yellow_card",
        "red_card": "is_red_card",
        "substitution_on": "is_substitution_on",
        "substitution_off": "is_substitution_off",
    }
    boolean_data = {final_col: as_bool(source_col) for final_col, source_col in final_boolean_sources.items()}
    boolean_data.update({
        "shot_zone_6_yard_box": processed_events["shot_zone"].eq("6-yard box"),
        "shot_zone_penalty_area": processed_events["shot_zone"].eq("Penalty Area"),
        "shot_zone_outside_box": processed_events["shot_zone"].eq("Outside of box"),
        "shot_open_play": processed_events["shot_situation"].eq("Open Play"),
        "shot_fastbreak": processed_events["shot_situation"].eq("Fastbreak"),
        "shot_set_piece": processed_events["shot_situation"].eq("Set Pieces"),
        "shot_penalty": processed_events["shot_situation"].eq("Penalty"),
        "shot_right_foot": processed_events["shot_body_part"].eq("Right foot"),
        "shot_left_foot": processed_events["shot_body_part"].eq("Left foot"),
        "shot_head": processed_events["shot_body_part"].eq("Head"),
        "shot_other_body_part": processed_events["shot_body_part"].eq("Other body parts"),
    })
    boolean_frame = pd.DataFrame(boolean_data, index=processed_events.index)
    processed_events = processed_events.drop(columns=[c for c in boolean_frame.columns if c in processed_events.columns])
    processed_events = pd.concat([processed_events, boolean_frame], axis=1)

    old_helper_flags = [c for c in processed_events.columns if c.startswith("is_")]
    old_helper_flags.extend(["shot_is_goal", "shot_assisted_key_pass", "pass_successful"])
    processed_events = processed_events.drop(columns=[c for c in old_helper_flags if c in processed_events.columns])

    ordered_cols = [
        # Match metadata
        "game_id", "season", "competition_country", "competition_name", "game_date", "game_status", "week",
        "home_team_id", "home_team_name", "away_team_id", "away_team_name",
        # Event identity and context
        "event_idx", "period", "minute", "second", "expanded_minute",
        "team_id", "team", "player_id", "player", "type", "outcome_type",
        "x", "y", "end_x", "end_y", "goal_mouth_y", "goal_mouth_z", "blocked_x", "blocked_y",
        "related_event_id", "related_player_id", "stat_event_type",
        # WhoScored top tiles
        "shot_event", "pass_event", "pass_completed", "dribble_event", "tackle_attempted_event",
        "interception_event", "clearance_event", "block_event",
        "offside_event", "foul_event", "aerial_duel_event", "touch_event", "loss_possession_event",
        "error_event", "save_event", "claim_event", "punch_event",
        "goalkeeper_event", "ball_recovery_event", "card_event", "substitution_event",
        # Shot details: Results, Zones, Situation, Body Parts
        "shot_goal", "shot_on_target", "shot_off_target", "shot_woodwork", "shot_blocked",
        "shot_own_goal",
        "shot_zone_6_yard_box", "shot_zone_penalty_area", "shot_zone_outside_box",
        "shot_open_play", "shot_fastbreak", "shot_set_piece", "shot_penalty",
        "shot_right_foot", "shot_left_foot", "shot_head", "shot_other_body_part",
        "shot_result", "shot_zone", "shot_situation", "shot_body_part",
        # Pass details: Type, Length, Height, Body Parts, Direction, Target Zone
        "pass_attempt", "pass_incomplete",
        "pass_cross", "pass_freekick", "pass_corner", "pass_through_ball", "pass_throw_in",
        "pass_key_pass", "pass_key_pass_qualifier",
        "pass_long", "pass_short", "pass_length",
        "pass_chipped", "pass_ground", "pass_height",
        "pass_head", "pass_feet", "pass_body_part",
        "pass_forward", "pass_backward", "pass_left", "pass_right",
        "pass_defensive_third", "pass_mid_third", "pass_final_third", "pass_target_zone",
        # Remaining stat details
        "dribble_successful", "dribble_unsuccessful",
        "tackle_gained_possession", "tackle_did_not_get_possession", "tackle_was_dribbled", "tackle_result",
        "clearance_head", "clearance_feet", "clearance_body_part",
        "blocked_shot", "blocked_cross", "block_type",
        "caught_offside", "offside_pass", "offside_provoked", "offside_type",
        "foul_committed", "foul_suffered", "foul_type",
        "aerial_duel_won", "aerial_duel_lost", "aerial_result",
        "dispossessed", "turnover", "loss_possession_type",
        "error_leading_to_shot", "error_leading_to_goal", "error_type",
        "keeper_pickup", "keeper_sweeper", "gk_type",
        "yellow_card", "second_yellow_card", "red_card", "card_type",
        "substitution_on", "substitution_off", "substitution_type",
    ]

    ordered_cols = [c for c in ordered_cols if c in processed_events.columns]
    remaining_cols = [c for c in processed_events.columns if c not in ordered_cols]
    processed_events = processed_events[ordered_cols + remaining_cols]
    return processed_events



In [10]:
from pymongo.errors import BulkWriteError

raw_events_collection = events_config.collection["collection_raw_events"]
processed_events_collection = events_config.collection["collection_processed_events"]


def normalize_mongo_value(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return {k: normalize_mongo_value(v) for k, v in value.items()}
    if isinstance(value, list):
        return [normalize_mongo_value(v) for v in value]
    if isinstance(value, tuple):
        return [normalize_mongo_value(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def records_for_mongo(df: pd.DataFrame) -> list[dict]:
    return [
        {k: normalize_mongo_value(v) for k, v in record.items()}
        for record in df.to_dict(orient="records")
    ]


def season_pending_game_ids() -> list[int]:
    client = MongoClient(events_config.url)
    db = client[events_config.db]
    raw_collection = db[raw_events_collection]
    processed_collection = db[processed_events_collection]

    raw_ids = {
        int(game_id)
        for game_id in raw_collection.distinct("game_id", {"season": events_config.year})
        if game_id is not None
    }
    processed_ids = {
        int(game_id)
        for game_id in processed_collection.distinct("game_id", {"season": events_config.year})
        if game_id is not None
    }
    client.close()
    return sorted(raw_ids - processed_ids)


pending_game_ids = season_pending_game_ids()
print(f"Season: {events_config.year}")
print(f"Pending season games: {len(pending_game_ids):,}")
print(pending_game_ids[:10])


Season: 2025-2026
Pending season games: 279
[1908319, 1910598, 1910599, 1910600, 1910601, 1910602, 1910603, 1910604, 1910605, 1910606]


## Run Season Save


In [ ]:
LIMIT = 1

client = MongoClient(events_config.url)
db = client[events_config.db]
raw_collection = db[raw_events_collection]
processed_collection = db[processed_events_collection]
game_ids_to_process = pending_game_ids if LIMIT is None else pending_game_ids[:LIMIT]
inserted_games = 0
inserted_rows = 0

for game_id in game_ids_to_process[:1]:
    raw_events = pd.DataFrame(
        raw_collection.find({"season": events_config.year, "game_id": game_id}, {"_id": 0})
    )
    if raw_events.empty:
        print(f"game_id={game_id}: no raw rows")
        continue

    processed_events = build_processed_game_events(raw_events)
    records = records_for_mongo(processed_events)
    if not records:
        print(f"game_id={game_id}: no processed rows")
        continue

    processed_collection.delete_many({"season": events_config.year, "game_id": game_id})
    try:
        result = processed_collection.insert_many(records, ordered=False)
    except BulkWriteError:
        processed_collection.delete_many({"season": events_config.year, "game_id": game_id})
        raise

    inserted_games += 1
    inserted_rows += len(result.inserted_ids)
    print(f"game_id={game_id}: inserted {len(result.inserted_ids):,} rows")

client.close()

{"processed_games": inserted_games, "inserted_rows": inserted_rows}
